# Week 5 — select, freeze and validate the profiles

**Question:** which development-country clustering gives stable, interpretable profiles, and how well do its fixed definitions transfer to other countries?

Follow the steps in order: (1) search worry coding and allocation, (2) confirm promising settings, (3) inspect individual profiles, (4) check shallow explanations, (5) freeze T5 and assess transfer. The appendix retains regional and country-distance analyses under B/E. Those are not T5 regional refits or T5 country-distance results.

The clustering uses complete responses, equal-country weights, separate personal/known-person harm, and no PCA or z-standardization. T5 is the five-profile solution used for the course continuation. The country split and profile IDs stay fixed.

This notebook reads saved results by default. Optional searches and fits remain off. The held-out countries had already been examined under B/E before T5 was selected: T5 transfer is a secondary assessment, not an untouched external confirmation.

## Start here

Run from the course folder or `weekly_notebooks`. The required inputs are in `course_artifacts` and `23_wrp`. Install the listed scientific packages if needed. No absolute personal paths are required.

Each section states its purpose, shows its code and results, and explains the decision. Saved fits are reused; opening the notebook does not launch a new search. Original notebooks and longer historical explanations remain in `earlier iterations`.

### Packages

With Python 3.11, install once from the course folder:
`python -m pip install -r weekly_notebooks/requirements-objective1.txt`

In [ ]:
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from threadpoolctl import threadpool_limits


### Optional refits — off by default

In [ ]:
RUN_ONE_LOCAL_REFERENCE = False


In [ ]:
# Run from the course folder or weekly_notebooks. No absolute data paths.
ROOT = Path.cwd()
while not (ROOT / 'course_artifacts').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'course_artifacts').is_dir(), 'Open the course folder first.'
ART = ROOT / 'course_artifacts'
OUT = ART / 'objective1_simplified_checks'
OUT.mkdir(exist_ok=True)


In [ ]:
DOMAINS = ['food', 'water', 'crime', 'severe_weather', 'traffic', 'mental_health']
WORRY = ['worry_' + d for d in DOMAINS]
HARM = ['harm_' + d for d in DOMAINS + ['work']]
SELF = [f + '_personal' for f in HARM]
KNOWN = [f + '_known_person' for f in HARM]
FEATURES = WORRY + SELF + KNOWN


In [ ]:
def country_weights(frame):
    # Each country has equal total mass; preserve relative survey weights within it.
    return (frame.WGT / frame.groupby('COUNTRY_ISO3').WGT.transform('sum')
            / frame.COUNTRY_ISO3.nunique()).to_numpy()


### Shared feature geometry

Recode worry as $0,1/t,1$. Let $m_i$ be respondent $i$'s mean over the six worries. Keep six deviations from that mean and the mean itself, multiplying both by $\sqrt{\alpha/6}$. Multiply the 14 binary harm indicators by $\sqrt{(1-\alpha)/14}$.

The mean is derived from the same six answers; it is not a seventh independent worry feature. No imputation is used. This transformation estimates no parameters from the evaluation data.

In [ ]:
def profile_matrix(frame, top, alpha):
    values = frame[FEATURES].to_numpy(float)
    assert np.isfinite(values).all(), 'This analysis uses complete responses only.'
    worry = np.where(values[:, :6] == 2, top, values[:, :6]) / top
    mean_worry = worry.mean(axis=1, keepdims=True)
    # Preserve the original floating-point operation order for cache comparisons.
    return np.column_stack([
        (worry - mean_worry) * np.sqrt(alpha) / np.sqrt(6),
        mean_worry * np.sqrt(alpha) / np.sqrt(6),
        values[:, 6:] * np.sqrt((1 - alpha) / 2) / np.sqrt(7),
    ])


## 1. Search worry coding and worry–harm allocation

**Purpose:** replace a few hand-picked representations with a bounded development-only comparison. Parameter $t$ controls the position of “somewhat worried” relative to “very worried”. Allocation $\alpha$ controls worry's share of the squared-distance budget.

Search $q=1/t$ from 0.1 to 0.9 and $\alpha$ from 0.2 to 0.8, with $k=3,\ldots,10$. The initial search uses 240 settings and 61 refinements. K-means itself is unchanged: 20 initializations and at most 300 iterations.

Retention requires largest equal-country cluster share ≤40%, dominance $D=k\max_j p_j<3$, and median **and lower-decile ARI ≥0.80 in both stability tests**. The lower decile is the 10th percentile. Respondent tests retain 80% within each country; country tests retain 90 of 113 countries, stratified by region and income. Fifty repetitions give 1,225 pairwise ARIs per test. Pilot scores only screen settings.

In [ ]:
PARAMETER_SEARCH = ROOT / 'course_artifacts/week04_kmeans_parameter_optimization/8ecb97fd957f69f8'
parameter_protocol = json.loads((PARAMETER_SEARCH / 'protocol.json').read_text())
display(pd.Series(parameter_protocol['design'], name='predeclared_search'))
print('Saved progress:', json.loads((PARAMETER_SEARCH / 'progress.json').read_text()))

### The K-means declaration

Only $q$, $\alpha$ and $k$ vary. The function below uses the same features and weighting for every setting.

In [ ]:
def fit_tuned_kmeans(frame, q, alpha, k, seed):
    matrix = profile_matrix(frame, top=1 / q, alpha=alpha)
    model = KMeans(n_clusters=k, n_init=20, max_iter=300, random_state=seed)
    with threadpool_limits(limits=1):
        model.fit(matrix, sample_weight=country_weights(frame))
    return model

### Optional search execution

Leave the switch off to read saved results. Enable it only for a deliberate recomputation or verified cache resume; do not start duplicate searches.

In [ ]:
RUN_PARAMETER_OPTIMIZATION = False
if RUN_PARAMETER_OPTIMIZATION:
    import subprocess
    import sys
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/run_search.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/report_results.py')], check=True)

### Saved search results

Read screening and full confirmation separately. A candidate is retained only from full confirmation, not because its pilot score looked promising.

In [ ]:
for filename in ['coarse_results.csv', 'full_confirmation_results.csv', 'best_tested_setting_per_k.csv']:
    print(filename)
    table = pd.read_csv(PARAMETER_SEARCH / filename)
    columns = [c for c in ['k', 'q', 't', 'alpha', 'respondents_median', 'respondents_p10',
               'countries_median', 'countries_p10', 'largest', 'dominance', 'stability_pass'] if c in table]
    display(table[columns].round(4))

In [ ]:
figure = PARAMETER_SEARCH / 'coarse_stability_surfaces.png'
if figure.exists():
    from IPython.display import Image
    display(Image(filename=str(figure)))

### Initial-grid conclusion

The initial grid supported resolutions $k=3,4,6,7,8$ under all numerical filters. Its tested $k=5,9,10$ settings did not pass. This says nothing about untested parameter values. The next bounded search checks values between grid points, without opening held-out responses or changing the retention thresholds.

## 2. Search between grid points, then confirm once

**Purpose:** test whether the grid missed useful settings. Import the 301 evaluated settings, then use a bounded adaptive search with 20 new parameter pairs for each $k$: 160 new settings in total. Ten repetitions per test screen them. Select two eligible settings per $k$, add unchanged B/E benchmarks, and use fresh 50-repetition confirmation samples.

Do not treat the adaptive pilot maximum as a final stability estimate. The same structure and ARI filters apply. There were no model-fitting failures. A preflight seed-overlap check stopped before new fitting and was corrected; it was not a failed candidate fit.

In [ ]:
ADAPTIVE_SEARCH = ROOT / 'course_artifacts/week04_kmeans_adaptive_optimization/54d309fe37735497'
adaptive_protocol = json.loads((ADAPTIVE_SEARCH / 'protocol.json').read_text())
display(pd.Series(adaptive_protocol['design'], name='adaptive_protocol'))
status_file = ADAPTIVE_SEARCH / 'completion.json'
print(json.loads(status_file.read_text()) if status_file.exists() else 'Search or confirmation still running; no final conclusions yet.')

### Optional adaptive execution

The switch remains off. The runner uses the fixed bounds, saved optimizer state, a single-run lock and checked fit caches. It does not load held-out responses.

In [ ]:
RUN_ADAPTIVE_OPTIMIZATION = False
if RUN_ADAPTIVE_OPTIMIZATION:
    import subprocess
    import sys
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/adaptive_search.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/report_adaptive.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/test_adaptive.py')], check=True)

### Confirmed numerical comparison

Compare settings on the same fresh samples. “Pass” requires every structural and stability condition. Show the best tested setting at each $k$, including failures, so the selection is transparent.

In [ ]:
adaptive_best = pd.read_csv(ADAPTIVE_SEARCH / 'best_tested_setting_per_k.csv')
baseline = pd.read_csv(ADAPTIVE_SEARCH / 'frozen_baselines_on_confirmation_samples.csv')
adaptive_best['model'] = 'T' + adaptive_best.k.astype(str)
baseline['model'] = baseline.baseline
comparison = pd.concat([adaptive_best, baseline], ignore_index=True).sort_values(['k', 'model'])
display(comparison[['model', 'k', 'q', 't', 'alpha', 'respondents_median', 'respondents_p10',
                    'countries_median', 'countries_p10', 'largest', 'dominance', 'stability_pass']].round(4))

In [ ]:
figure = ADAPTIVE_SEARCH / 'adaptive_search_surfaces.png'
if figure.exists():
    from IPython.display import Image
    display(Image(filename=str(figure)))

### Adaptive-search conclusion

The search added 160 settings to the earlier 301. Fresh confirmation supported at least one setting for every $k=3,\ldots,9$; the best tested $k=10$ failed. This does not establish a global optimum.

The five-profile candidate has $t=2.399802888616365$ and $\alpha=0.5837857862168715$. Its respondent ARI median/lower decile are 0.974/0.946; country values are 0.956/0.913. Its largest cluster is 36.3%, so $D=1.816<3$.

Do not select solely by the best ARI. Coarser solutions can be more stable, while a high whole-partition score can conceal a weak individual profile. Inspect meanings and profile recovery next.

## 3. Check meanings and individual-profile recovery

**Purpose:** compare the seven numerically eligible adaptive selections and B4/E8 using existing fits. No new clustering is needed for these comparisons.

In [ ]:
PROFILE_EVALUATION = ROOT / 'course_artifacts/week04_kmeans_profile_evaluation/59d97cff396076a4'
display(pd.Series(json.loads((PROFILE_EVALUATION / 'completion.json').read_text())))

### 3.1. Describe the original answers

For every fixed profile, report equal-country prevalence, respondent prevalence, worry-category distributions, personal harm and known-person harm. The descriptions should show what distinguishes the profiles, not just their transformed centre coordinates.

In [ ]:
profile_cards = pd.read_csv(PROFILE_EVALUATION / 'profile_review_cards.csv')
display(profile_cards[['model', 'cluster', 'description', 'weighted_share', 'respondent_share',
                       'mean_worry_0_to_2', 'mean_personal_harm_prevalence',
                       'mean_known_person_harm_prevalence']].round(3))
from IPython.display import Image
display(Image(filename=str(PROFILE_EVALUATION / 'original_unit_profile_heatmap.png')))
display(Image(filename=str(PROFILE_EVALUATION / 'original_worry_category_distributions.png')))

### 3.2. Can each profile be recovered?

Whole-partition ARI is not an individual-profile test. Match each fixed reference cluster to repetition clusters and measure Jaccard overlap (intersection divided by union). One-to-one matching avoids explaining two reference clusters with the same repetition cluster.

Inspect the weakest profile's lower-decile recovery in both tests. The best-any-match and reference-objective diagnostics help distinguish genuinely fragile profiles from a reference partition that is less representative than repeated fits.

In [ ]:
review = pd.read_csv(PROFILE_EVALUATION / 'model_review_summary.csv')
display(review[['model', 'k', 'score', 'respondents_min_cluster_jaccard_p10',
                'countries_min_cluster_jaccard_p10', 'respondents_minimum_best_any_jaccard_p10',
                'countries_minimum_best_any_jaccard_p10', 'cached_centres_with_lower_full_objective']].round(3))
display(Image(filename=str(PROFILE_EVALUATION / 'individual_profile_stability.png')))
for filename in ['reference_representativeness.csv', 'reference_inertia_diagnostic.csv']:
    print(filename)
    display(pd.read_csv(PROFILE_EVALUATION / filename).round(4))

### 3.3. What changes at a finer resolution?

Compare membership overlaps and original answers. A larger $k$ is useful only if it adds a reproducible substantive distinction. Overlapping partitions are not necessarily nested, so the comparison is an approximate refinement graph, not a mathematical lattice.

In [ ]:
display(Image(filename=str(PROFILE_EVALUATION / 'cross_resolution_membership_overlaps.png')))
pair_comparisons = pd.read_csv(PROFILE_EVALUATION / 'partition_comparisons.csv')
display(pair_comparisons.loc[(pair_comparisons.source.isin(['B4', 'E8', 'T5', 'T7', 'T8'])) &
                            (pair_comparisons.target.isin(['T4', 'T5', 'E8', 'T7', 'T8', 'T9']))].round(3))
added_profiles = pd.read_csv(PROFILE_EVALUATION / 'added_profile_review.csv')
display(added_profiles.loc[added_profiles.source.eq('T8') & added_profiles.target.eq('T9')].round(3))

### Decision from profile-level evidence

T5 gives five recognizable groups: lower worry; broad known-person harm; crime/weather worry; mental-health/traffic/weather worry; water/food worry. Its weakest country-test profile has lower-decile Jaccard 0.943.

T3 and T4 are more stable but describe coarser contrasts. Higher resolutions pass global ARI yet have much weaker individual-profile recovery: the weakest country-test lower-decile Jaccards are approximately 0.419 (T6), 0.452 (T7), 0.471 (T8) and 0.066 (T9). E8 is 0.087. These scores caution against adopting a finer profile taxonomy solely from global ARI.

Take T5 forward as a defensible moderate resolution, not a uniquely optimal clustering. Keep B/E and other evaluated settings as documented comparisons. First check whether shallow rules nearly reproduce T5.

### Optional diagnostic reproduction

The code below reads existing fit caches and recomputes comparison diagnostics. It does not fit new profile models. Leave the switch off for ordinary reading.

In [ ]:
RUN_CACHED_PROFILE_EVALUATION = False
if RUN_CACHED_PROFILE_EVALUATION:
    import subprocess
    import sys
    for filename in ['evaluate_profiles.py', 'diagnose_reference_profiles.py',
                     'summarize_profile_evaluation.py', 'test_profile_evaluation.py']:
        subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search' / filename)], check=True)
print(json.loads((PROFILE_EVALUATION / 'verification.json').read_text()))

## 4. Check whether T5 has a trivial explanation

**Purpose:** ask whether the five fixed labels merely restate overall burden or one original answer. This reconstructs existing membership; it does not validate an independently discovered clustering.

Use all five country-grouped folds and equal-country-weighted balanced accuracy. Folds choose reconstruction rules using training countries and evaluate them in different countries.

### 4.1. Burden and single-feature checks

Burden trees use only mean original worry and mean harm exposure. Single-feature checks predict membership versus nonmembership by searching original features, category thresholds and both directions in training countries. Report balanced accuracy, precision and recall; high recall alone is insufficient.

In [ ]:
T5_EXPLANATIONS = ROOT / 'course_artifacts/week04_t5_shallow_explanations/4f0f385fb4cbe644'
display(pd.read_csv(T5_EXPLANATIONS / 'burden_tree_summary.csv').round(4))
display(pd.read_csv(T5_EXPLANATIONS / 'single_feature_summary.csv').round(4))

### 4.2. Shallow trees with all 20 features

Test tree depths 1, 2, 3 and 5 with the same country folds and minimum leaf size 200. This checks multifeature shortcuts that the burden and single-feature checks can miss.

In [ ]:
display(pd.read_csv(T5_EXPLANATIONS / 'all_feature_tree_summary.csv').round(4))
t5_tree_profiles = pd.read_csv(T5_EXPLANATIONS / 'all_feature_tree_per_cluster_summary.csv')
display(t5_tree_profiles.loc[t5_tree_profiles.depth.isin([2, 3])].round(4))
print('Example: depth-3 tree fitted without the first evaluation-country fold')
print((T5_EXPLANATIONS / 'depth3_fold1_rules.txt').read_text())

### Reconstruction conclusion

Burden-tree balanced accuracies at depths 1/2/3 are 32.7%/46.0%/51.2%. All-feature trees reach 36.8%/54.0%/67.9%; depth 5 reaches 73.7%.

A depth-1 tree has at most two leaves, so its maximum balanced accuracy for five classes is 40%; depth 2 is limited to 80%. Interpret scores against that capacity, not against an unattainable 100%.

Very worried about water is the strongest single-feature description of C4: 88.3% balanced accuracy, 67.4% precision and 84.3% recall. It is characteristic, not an exact membership rule. The checks do not nearly reproduce all five groups from a simple burden rule or a single answer. Retain T5.

## 5. Freeze T5 and assess held-out transfer

**Purpose:** preserve the exact development centres and apply them unchanged to respondents in the 29 held-out countries. There is no held-out or all-country clustering refit. Because these countries were previously examined under B/E, report secondary fixed-model transfer, not untouched external validation.

In [ ]:
T5_TRANSFER = ROOT / 'course_artifacts/week05_t5_secondary_transfer/832b01713bc6b240'
T5_PACKAGE = T5_TRANSFER / 'frozen_model'
t5_seal = json.loads((T5_PACKAGE / 'freeze_seal.json').read_text())
for filename, expected in t5_seal['files'].items():
    assert hashlib.sha256((T5_PACKAGE / filename).read_bytes()).hexdigest() == expected
t5_specification = json.loads((T5_PACKAGE / 'specification.json').read_text())
settings = dict(t5_specification['spec'])
settings['status'] = 'frozen; transfer completed'
display(pd.Series(settings))
display(pd.Series(json.loads((T5_TRANSFER / 'transfer_complete.json').read_text())).drop('files'))

### 5.1. Verify the frozen model

Check the saved file hashes, centre order and development labels. Preserve T5_C0–T5_C4. Complete-response eligibility, coding, geometry, nearest-centre assignment, exact-tie convention and development distance thresholds stay fixed.

Prepared respondents: 116,820 development and 30,090 held-out. Assigned complete cases: 109,488 and 27,972. The remaining 9,450 respondents stay unassigned.

### 5.2. Compare frequencies, meanings and distances

Use equal-country weights within each reporting population. Compare original response categories and the three fixed defining contrasts per profile. A frequency change alone does not invalidate a profile.

Flag assignments beyond the assigned profile's fixed development weighted 95th-percentile distance. This marks less typical responses; it is not an error probability or an exclusion rule. The relative margin is (second-nearest distance − nearest distance) / second-nearest distance.

In [ ]:
display(pd.read_csv(T5_TRANSFER / 'profile_transfer_summary.csv').round(4))
from IPython.display import Image
display(Image(filename=str(T5_TRANSFER / 't5_prevalence_transfer.png')))
display(Image(filename=str(T5_TRANSFER / 't5_original_response_transfer.png')))
t5_anchors = pd.read_csv(T5_TRANSFER / 'heldout_frozen_anchor_comparison.csv')
display(t5_anchors.loc[t5_anchors.scope.eq('global'),
    ['cluster', 'feature', 'category', 'development_proportion', 'proportion',
     'development_contrast', 'difference_vs_outside', 'heldout_contrast_direction_preserved']].round(4))
display(pd.read_csv(T5_TRANSFER / 'heldout_vs_development_quality.csv').round(4))

### 5.3. Inspect country-specific weaknesses

Read distance flags together with complete-case retention and country/profile sample size. The semantic-distance tables use mean per-feature total variation: half the sum of absolute differences in category probabilities, averaged over features.

In [ ]:
t5_country = pd.read_csv(T5_TRANSFER / 'heldout_country_quality_and_retention.csv')
display(t5_country.sort_values('profile_p95_exceedance', ascending=False)[
    ['Country', 'prepared', 'complete', 'retained_fraction', 'profile_p95_exceedance',
     'distance_median', 'margin_median']].round(4))
t5_country_profiles = pd.read_csv(T5_TRANSFER / 'heldout_profile_semantic_differences.csv')
display(t5_country_profiles.loc[t5_country_profiles.scope.eq('country')].sort_values(
    'mean_feature_total_variation', ascending=False).head(15).round(4))

### Transfer conclusion

All 15 defining contrasts keep their direction in aggregate. Mean per-feature category differences are 0.008–0.028 across profiles. The largest prevalence shift is +2.1 percentage points for the mental-health/traffic/weather profile.

Distance flags rise from 4.95% to 5.69%; median nearest-centre distance is 0.2722 versus 0.2735, and median relative margin is 0.1515 versus 0.1524. Aggregate descriptions transfer reasonably well under fixed assignment, but that is not independent rediscovery of the profiles.

Country limits matter. Sierra Leone has 18.8% flagged and 78.7% complete-case retention; Namibia 16.9% flagged; Somalia 16.7% flagged and 82.9% retention. Sweden's water/food cell has only 15 respondents (weighted effective size 9.8). Somalia's corresponding semantic shift is based on 173 respondents, so it cannot be dismissed as the same tiny-cell problem.

The 1,000 country-bootstrap intervals hold predictions and model fixed. They describe variation over these 29 countries, not all-world sampling, missingness or model-selection uncertainty. No automatic transfer-rejection cutoff was specified. The conclusion is broad aggregate descriptive transfer, qualified by country and complete-case limitations. Continue with frozen T5.

### 5.4. Reuse assignments by respondent ID

The registry retains all 146,910 IDs, the original split, eligibility and exclusions. Never join by row order, impute missing labels or renumber profiles by subgroup prevalence.

The student package in `student_materials/T5_frozen_profiles` contains the compact mapping and instructions. Keep respondent-linked files out of public GitHub. Week 6 uses T5 labels as a categorical prediction target, not as an extra predictor.

In [ ]:
print(json.loads((T5_TRANSFER / 'verification.json').read_text()))
display(pd.Series(json.loads((T5_TRANSFER / 'primary_model_decision.json').read_text())))
# Reuse by ID, not by row position; the original survey dataset is not overwritten.
t5_handoff = pd.read_parquet(T5_TRANSFER / 'all_respondents_T5_eligibility_and_frozen_profiles.parquet')
assert t5_handoff.WPID_RANDOM.is_unique
display(t5_handoff.groupby('country_split').agg(
    prepared=('WPID_RANDOM', 'size'), complete_and_assigned=('complete_case', 'sum')))

### Objective 1 handoff

T5 is selected, frozen and assessed. The original data are not overwritten. The code below and the saved artifacts retain the numerical evidence. The appendix distinguishes earlier B/E regional and country-distance analyses from T5 results; no T5 regional refits or country-distance estimates are implied.

## 6. Complete the geographical and country analyses under T5

T5 regional completion: 6 cases pass both full stability tests; 0 failed stability fits. Small-region LOO and respondent-only diagnostics are not full geographic confirmation. Global T5 was not refitted.

Regional selection holds the T5 geometry fixed, examines k=2–10, and tests k=5 plus coarse and inertia-based alternatives. The 0.80 median/lower-decile ARI rule is unchanged. Structural global benchmarks are reported, not imposed as universal regional filters. Compare regional meaning and overlaps; do not equate different local labels across regions.

Country distances use common global T5 profiles or original marginal categories, not unmatched local cluster numbers. Hellinger uses profile proportions; total variation uses alpha=0.5838 and is deliberately insensitive to numeric worry coding t. Both are post-validation descriptions. Bootstrap results condition on fixed assignments and do not represent survey-design or model-selection uncertainty. Income-label permutations move whole countries, not independent pairs.

The fixed all-country cross-model median denominators are retained, so adding T5 does not recolour historical B/E results. Map projection quality measures the loss from approximate two-dimensional colours; exact rankings use the original distance.



### Interpretation of the completed T5 geographical analyses

None of the 11 regions with the full 50-fit country test has a five-cluster solution passing both stability tests. Four other regions have limited or unavailable country tests. The supported full-test resolutions are Eastern Africa k=3; Central/Western Africa k=3 and 4; Latin America/Caribbean k=3 and 4; Northern/Western Europe k=2. Northern Africa k=4 passes respondent and limited leave-one-country-out diagnostics only. The one-country development regions have no geographic-stability test. Multiple supported resolutions are alternatives, not proof of a unique local optimum. Global size/imbalance rules are reported as benchmarks, not imposed as universal regional filters.

This is a material limit on a universal interpretation of the five global profiles. T5 is a stable, useful common global description, but independent regional optimization does not reproduce five reliable groups everywhere. That does not invalidate applying fixed global profiles within regions: fixed assignment and independent regional discovery answer different questions. Neither justifies claiming five universal sociological types.

Some stable local solutions are simple. The Northern/Western European k=2 solution separates a group with 98.7% known-person mental-health harm from a group with 0%. Its stability should not be presented as evidence of a rich multifeature regional taxonomy. Supported local profiles and their original-unit contrasts are saved separately; local numeric IDs have no common meaning across regions. Correspondence to T5 uses observed membership overlap and category differences, not an assumption that local profiles equal the global ones.

Under fixed global T5 assignment, lower-worry prevalence is 56.4% in Northern/Western Europe versus 17.1% in Central/Western Africa and 13.8% in Southern Africa. Broad known-person harm is 8.2% versus 23.4% in the first two regions. These are equal-country summaries of complete-case respondents, not world-population prevalence estimates. The regional original-category tables show how meanings and frequencies vary.

Country similarity is only partly geographical. Mean Hellinger distance is 0.174 to same-region peers versus 0.233 to other-region peers, averaging countries with regional peers. The nearest country is in the same region for 23.9%, compared with an 8.1% random-peer reference. Physical map-label distance has rank correlation 0.152 with profile-mixture distance. The exploratory best-silhouette country cut has two groups (97 and 45 countries), with region-agreement ARI 0.052. It is not adopted as a replacement regional classification.

Distances depend on what is summarized. The USA's nearest country by T5 profile mixture is Bosnia Herzegovina; Iran is third. Finland's nearest is Tajikistan. By original-response distance with T5 allocation, Canada is first for the USA and Iceland first for Finland. These differences show information lost when response distributions are compressed into five profile shares. Familiar neighbours do not establish validity, and unexpected neighbours are not grounds to tune the distance. The two summaries answer different questions and neither measures societal equivalence.

Nearest ranks are uncertain even with the profiles fixed: the reference nearest neighbour recurs in only 29.5% of conditional respondent bootstraps for both the USA and Finland; the median across countries is 40.5%. These are IID respondent sensitivity checks retaining WGT, not survey-design confidence or profile-selection uncertainty. Lists should be read as proximity descriptions, not certain top-ranked relationships.

Across 141 income-classified countries, income-gap Spearman correlation is 0.2469 for profile-mixture Hellinger distance and 0.2977 for original-response distance. Within-region whole-country income-label permutations give upper-tail fractions 0.006 and 0.003. These conditional exchangeability references do not establish causation or control for all country differences; pairs are not treated as independent observations.

Two overall burden totals cannot exactly recover membership: even the within-sample optimal lookup by original worry sum and affected-domain count reaches only 60.1% accuracy in development and 58.2% in held-out data. This is a descriptive information-loss bound, not cross-validated predictive performance or proof of added value for an external outcome. Objective 3 still requires that separate test.


In [ ]:
T5_COMPLETION = ART / 'week05_T5_completion/70ee2f67e95369c5'
for filename in ['supported_local_resolutions.csv','matched_regional_semantic_correspondence.csv','T5_fixed_profiles_by_split_and_region.csv']:
    print(filename)
    display(pd.read_csv(T5_COMPLETION / filename).round(4))
for filename in ['country_geography_summary.csv','income_distance_summary.csv','income_country_label_permutations.csv','profile_burden_and_heterogeneity.csv','burden_only_information_loss.csv','country_tree_cut_diagnostics.csv','conditional_nearest_neighbour_bootstrap.csv']:
    print(filename)
    display(pd.read_csv(T5_COMPLETION / 'countries' / filename).round(4))
display(Image(filename=str(T5_COMPLETION / 'figures/T5_local_resolution_stability.png')))
display(Image(filename=str(T5_COMPLETION / 'figures/T5_region_prevalence.png')))
for iso in ['FIN','USA']:
    display(Image(filename=str(T5_COMPLETION / 'countries' / f'T5_distance_from_{iso}_regions.png')))
display(pd.read_csv(T5_COMPLETION / 'supported_local_semantic_correspondence.csv').round(4))

## Appendix A. Regional and country comparisons under B/E

Keep these completed analyses as comparisons. They are not rerun under T5 in this notebook. B4 uses balanced worry/harm allocation and 0/1/2 worry; E8 uses worry emphasis and 0/1/3 worry.

### A.1. Verify the frozen B/E definitions

Check the seal and use stored centre order. These historical model IDs remain separate from T5.

In [ ]:
FROZEN = ART / 'week05_frozen_objective1_v1/123d248adda6115f'
HELDOUT = ART / 'week05_heldout_transfer_v1/123d248adda6115f'
REGIONAL = ART / 'week05_regional_validation_v3/7c24b461bbb895b5'
LOCAL = ART / 'week05_local_resolutions_v4/f3145c704f52c156'
HANDOFF = ART / 'week05_objective1_handoff_v1/123d248adda6115f'
seal = json.loads((FROZEN / 'freeze_seal.json').read_text())
assert seal['status'] == 'frozen_before_heldout_responses'
for filename, digest in seal['files'].items():
    assert hashlib.sha256((FROZEN / filename).read_bytes()).hexdigest() == digest, filename
bundle = joblib.load(FROZEN / 'models.joblib')
assert bundle['context'] == seal['context']
assert bundle['feature_order'] == FEATURES
for name, model in bundle['models'].items():
    print(name, model['spec'])
    display(pd.Series(model['profile_names'], name='profile_name').rename_axis('cluster'))


### A.2. Replay assignments without fitting

Derive personal and known-person harm from original codes, retain complete cases, transform features and assign the nearest stored centre. Compare every result to the saved ID-linked registry.

In [ ]:
prepared = pd.read_parquet(ART / 'week03_2023_analysis_data.parquet')
for feature, source in zip(HARM, ['WP22442', 'WP22443', 'WP22444', 'WP22445', 'WP22446', 'WP22447', 'WP22448']):
    values = prepared[source]
    assert values.dropna().isin([1, 2, 3, 4, 98, 99]).all()
    prepared[feature + '_personal'] = values.map({1: 1., 2: 0., 3: 1., 4: 0.})
    prepared[feature + '_known_person'] = values.map({1: 0., 2: 1., 3: 1., 4: 0.})
complete = prepared[FEATURES].notna().all(axis=1)
data = prepared.loc[complete].reset_index(drop=True)
assert data.WPID_RANDOM.is_unique and len(data) == 137460
registry = pd.read_parquet(HANDOFF / 'all_respondents_eligibility_and_frozen_profiles.parquet').set_index('WPID_RANDOM')
assert prepared.WPID_RANDOM.is_unique and set(prepared.WPID_RANDOM) == set(registry.index)
assert np.array_equal(complete, registry.loc[prepared.WPID_RANDOM, 'complete_case'].to_numpy())
assignment = data[['WPID_RANDOM', 'COUNTRY_ISO3', 'GlobalRegion', 'CountryIncomeLevel', 'country_split', 'WGT']].copy()
for name, model in bundle['models'].items():
    spec = model['spec']
    values = data[FEATURES].to_numpy(float)
    worry = np.where(values[:, :6] == 2, spec['top'], values[:, :6]) / spec['top']
    mean = worry.mean(axis=1, keepdims=True)
    X = np.column_stack([(worry - mean) * np.sqrt(spec['alpha'] / 6),
                         mean * np.sqrt(spec['alpha'] / 6),
                         values[:, 6:] * np.sqrt((1 - spec['alpha']) / 14)])
    distance = cdist(X, model['centres'])
    labels = distance.argmin(axis=1)
    nearest = np.sort(distance, axis=1)[:, :2]
    assignment[name + '_cluster'] = labels
    assignment[name + '_distance'] = nearest[:, 0]
    assignment[name + '_margin'] = np.divide(nearest[:, 1] - nearest[:, 0], nearest[:, 1],
                                            out=np.zeros(len(data)), where=nearest[:, 1] > 0)
    assignment[name + '_flag'] = nearest[:, 0] > np.array([model['profile_distance_p95'][int(j)] for j in labels])
    assert np.array_equal(labels, registry.loc[data.WPID_RANDOM, name + '_cluster'].to_numpy())
    assert np.allclose(nearest[:, 0], registry.loc[data.WPID_RANDOM, name + '_nearest_distance'])
print('All 274,920 B/E assignments reproduced exactly; no clustering fit.')
display(prepared.assign(complete=complete).groupby('country_split').complete.agg(['size', 'sum', 'mean']))


### A.3. Describe fixed profiles within geographical areas

The same global centres define the profiles in every region. Report regional frequencies and original answers using equal-country weights within the region. This does not optimize a new clustering in each region.

In [ ]:
def prevalence(frame, labels):
    mass = country_weights(frame)
    result = pd.DataFrame({'respondents': pd.Series(labels).value_counts().sort_index(),
                           'equal_country_share': pd.Series(mass).groupby(labels).sum()})
    result['respondent_share'] = result.respondents / len(frame)
    return result.rename_axis('cluster')


In [ ]:
for split, group in assignment.groupby('country_split'):
    for name in ['B', 'E']:
        print(split, name)
        display(prevalence(group, group[name + '_cluster'].to_numpy()).round(3))
regional_rows = []
for (split, region), group in assignment.groupby(['country_split', 'GlobalRegion']):
    for name in ['B', 'E']:
        table = prevalence(group, group[name + '_cluster'].to_numpy()).reset_index()
        table['split'], table['region'], table['candidate'] = split, region, name
        regional_rows.append(table)
regional_prevalence = pd.concat(regional_rows, ignore_index=True)
regional_prevalence.to_csv(OUT / 'week05_recomputed_regional_prevalence.csv', index=False)
display(regional_prevalence.round(3))
# The completed evidence also includes every individual country and original category.
display(pd.read_csv(REGIONAL / 'fixed_global_profile_prevalence.csv').head(24))
display(pd.read_csv(REGIONAL / 'fixed_global_original_profiles_by_region.csv').head(24))


### A.4. Independent regional clustering

This asks a different question: which groups best describe a particular region? Examine k=2–10 under B/E and retain the same median and lower-decile ARI threshold 0.80 in both tests. Fourteen cases pass: seven have two groups and seven have at least three. Small-region diagnostics are reported separately.

Do not lower the stability threshold because a preferred finer resolution failed. Different regional optima can be informative without invalidating the common global description.

In [ ]:
regional_diagnostics = pd.read_csv(LOCAL / 'all_local_resolution_diagnostics.csv')
local_support = pd.read_csv(LOCAL / 'supported_local_resolutions.csv')
display(local_support.round(3))
display(pd.read_csv(LOCAL / 'qualified_local_profile_interpretations.csv'))
display(pd.read_csv(LOCAL / 'regional_supported_k_sets.csv'))


In [ ]:
# Reproduce the bounded extra-k shortlist from inertia, not local/global agreement.
shortlist = []
for (region, name), group in regional_diagnostics.groupby(['region', 'candidate']):
    group = group.sort_values('k').copy()
    inertia = group.inertia.to_numpy()
    group['gap'] = (1 - (group.k - 2) / 8) - (inertia - inertia[-1]) / (inertia[0] - inertia[-1])
    matched_k = bundle['models'][name]['spec']['global_k']
    eligible = group.loc[group.k.between(3, 9) & group.k.ne(matched_k)]
    elbow = int(eligible.loc[eligible.gap.ge(.95 * eligible.gap.max()), 'k'].min())
    shortlist.extend((int(region), name, k) for k in sorted({2, 3, elbow} - {matched_k}))
saved_shortlist = pd.read_csv(LOCAL / 'additional_local_shortlist.csv')
assert set(shortlist) == set(saved_shortlist[['region', 'candidate', 'k']].itertuples(index=False, name=None))


In [ ]:
# An optional direct local fit; it writes nothing to the frozen package.
if RUN_ONE_LOCAL_REFERENCE:
    region, name, k = 1, 'B', 3
    seed = 20261011 + region
    group = data.loc[data.country_split.eq('train') & data.GlobalRegion.eq(region)]
    spec = bundle['models'][name]['spec']
    values = group[FEATURES].to_numpy(float)
    worry = np.where(values[:, :6] == 2, spec['top'], values[:, :6]) / spec['top']
    mean = worry.mean(axis=1, keepdims=True)
    X_local = np.column_stack([(worry - mean) * np.sqrt(spec['alpha']) / np.sqrt(6),
                               mean * np.sqrt(spec['alpha']) / np.sqrt(6),
                               values[:, 6:] * np.sqrt((1 - spec['alpha']) / 2) / np.sqrt(7)])
    with threadpool_limits(limits=1):
        local_model = KMeans(n_clusters=k, n_init=20, max_iter=300, random_state=seed)
        local_model.fit(X_local, sample_weight=country_weights(group))
    print('Illustrative new local fit only; not a replacement for saved local stability tests.')


### A.5. Historical B/E held-out transfer

Apply frozen profiles; compare original meanings, prevalence and assignment quality. The following tables remain B/E evidence, not T5 evidence.

In [ ]:
heldout_prevalence = pd.read_csv(HELDOUT / 'heldout_prevalence.csv')
display(heldout_prevalence.loc[heldout_prevalence.scope.eq('global')].round(3))
display(pd.read_csv(HELDOUT / 'heldout_vs_development_prevalence.csv').round(3))
display(pd.read_csv(HELDOUT / 'heldout_frozen_anchor_comparison.csv').round(3))
display(pd.read_csv(HELDOUT / 'heldout_country_bootstrap_intervals.csv').head(30).round(3))
for name in ['B', 'E']:
    flags = []
    for split, group in assignment.groupby('country_split'):
        flags.append({'candidate': name, 'split': split,
                      'weighted_flag_fraction': np.average(group[name + '_flag'], weights=country_weights(group))})
    display(pd.DataFrame(flags).round(4))
display(pd.read_csv(HELDOUT / 'heldout_assignment_quality.csv').head(30).round(3))


In [ ]:
# Recalculate original-category proportions for the global development/held-out profiles.
profile_rows = []
for split, group in data.groupby('country_split'):
    mass = country_weights(group)
    for name in ['B', 'E']:
        labels = assignment.loc[group.index, name + '_cluster'].to_numpy()
        for cluster in np.unique(labels):
            member = labels == cluster
            for feature in FEATURES:
                for category in range(3 if feature in WORRY else 2):
                    proportion = np.average(group.loc[member, feature].eq(category), weights=mass[member])
                    profile_rows.append({'split': split, 'candidate': name, 'cluster': int(cluster),
                                         'feature': feature, 'category': category, 'proportion': proportion})
profiles = pd.DataFrame(profile_rows)
profiles.to_csv(OUT / 'week05_recomputed_original_profiles.csv', index=False)
for split, filename in [('train', FROZEN / 'development_original_category_profiles.csv'),
                         ('heldout_country', HELDOUT / 'heldout_original_category_profiles.csv')]:
    original = pd.read_csv(filename)
    original = original.loc[original.scope.eq('global')]
    compared = profiles.loc[profiles.split.eq(split)].merge(original, on=['candidate', 'cluster', 'feature', 'category'],
                                                           suffixes=('_new', '_saved'), validate='one_to_one')
    assert len(compared) == 552 and np.allclose(compared.proportion_new, compared.proportion_saved, atol=1e-13)
print('All 1,104 global original-category profile proportions reproduced.')


### A.6. Country distances and maps

Hellinger distance compares country proportions over fixed profiles. Original-response distance compares original-category probabilities, averaging total variation separately over worry, personal harm and known-person harm. B/E use their stated allocations. Both summaries lose information about joint response patterns.

Display original-response distances relative to one fixed all-country median, 0.1550977480973838, pooled across B/E distinct-country pairs. Use separate fixed medians for components. A value 1 is the typical reference distance; values are uncapped, while the colour legend ends at “2 or more”. Selecting a different country or scope does not change the denominator.

Country similarity only partly follows dataset regions. Income-distance correlations are exploratory and country pairs are dependent. Neither proximity nor correlation establishes social equivalence or causality. The current interface is linked from the saved project webpage.

In [ ]:
DIST = ART / 'week05_country_response_distances_v1/e1ecf01551bcf2da'
probabilities = pd.read_csv(DIST / 'country_response_probabilities.csv')
countries = sorted(probabilities.COUNTRY_ISO3.unique())
feature_distances = []
country_mass = data.WGT / data.groupby('COUNTRY_ISO3').WGT.transform('sum')
for feature in FEATURES:
    table = probabilities.loc[probabilities.feature.eq(feature)].pivot(
        index='COUNTRY_ISO3', columns='category', values='proportion').loc[countries]
    # Reconstruct probabilities from respondents, not just from the saved matrix.
    current = data.assign(mass=country_mass).groupby(['COUNTRY_ISO3', feature]).mass.sum().unstack(fill_value=0)
    current = current.reindex(index=countries, columns=table.columns, fill_value=0)
    assert np.allclose(current, table, atol=1e-14)
    p = current.to_numpy()
    assert np.allclose(p.sum(axis=1), 1) and (p >= 0).all()
    feature_distances.append(np.abs(p[:, None, :] - p[None, :, :]).sum(axis=2) / 2)
distances = {'worry': np.mean(feature_distances[:6], axis=0),
             'self': np.mean(feature_distances[6:13], axis=0),
             'known': np.mean(feature_distances[13:], axis=0)}
for name, alpha in [('B', .5), ('E', .75)]:
    distances[name] = alpha * distances['worry'] + (1 - alpha) * (distances['self'] + distances['known']) / 2
for name, matrix in distances.items():
    original = pd.read_csv(DIST / f'{name}_all_countries_post_validation_matrix.csv', index_col=0).loc[countries, countries]
    assert np.allclose(matrix, original, atol=1e-14)
pairs = np.triu_indices(len(countries), 1)
median = np.median(np.concatenate([distances[name][pairs] for name in ['B', 'E']]))
assert np.isclose(median, .1550977480973838, atol=1e-14)
print('Pooled B/E distance median:', median)
display(pd.read_csv(DIST / 'response_income_summary.csv').round(3))


In [ ]:
# Also reconstruct the older profile-mixture Hellinger distance for both models.
mixture_source = ART / 'week05_country_similarity_and_profile_synthesis_v1/d694096841b06e54'
vectors = pd.read_csv(mixture_source / 'country_profile_vectors.csv')
for name in ['B', 'E']:
    table = assignment.assign(mass=country_mass).groupby(['COUNTRY_ISO3', name + '_cluster']).mass.sum().unstack(fill_value=0)
    table = table.reindex(index=countries, columns=range(bundle['models'][name]['spec']['global_k']), fill_value=0)
    saved_vectors = vectors.loc[vectors.candidate.eq(name)].pivot(index='COUNTRY_ISO3', columns='cluster', values='profile_share')
    assert np.allclose(table, saved_vectors.loc[table.index, table.columns], atol=1e-14)
    hellinger = cdist(np.sqrt(table), np.sqrt(table)) / np.sqrt(2)
    assert np.allclose(np.diag(hellinger), 0)
    print(name, 'profile-mixture Hellinger range:', hellinger.min(), hellinger.max())


In [ ]:
# Illustration: five nearest countries to the USA.
for name in ['B', 'E']:
    i = countries.index('USA')
    nearest = sorted((j for j in range(len(countries)) if j != i), key=lambda j: (distances[name][i, j], countries[j]))[:5]
    display(pd.DataFrame({'model': name, 'country': [countries[j] for j in nearest],
                          'raw_distance': [distances[name][i, j] for j in nearest],
                          'median_relative_distance': [distances[name][i, j] / median for j in nearest]}))


### Appendix conclusion

Use frozen T5 for the course continuation, retaining B/E comparisons under their original names. There is no new all-country fit. Original responses, assignment files and fit caches remain local. Country distances, local refits and prediction of membership answer different questions and must not be treated as interchangeable validation tests.